# 01 — Local LLM Foundation

Verify the local Ollama environment and the reusable utilities in the
`educational_shorts` package.

This notebook checks:

1. Project and package imports
2. Ollama installation and model availability
3. Structured Pydantic output from the local LLM
4. Prompt-file loading

The reusable implementation belongs in the package. This notebook only
configures and validates it.


## Project setup

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

Project root: c:\Users\hitch\python_files\educational_shorts


## Imports and configuration

In [2]:
import shutil
import subprocess

from pydantic import BaseModel

from educational_shorts.client import ask_llm
from educational_shorts.config import MODEL
from educational_shorts.prompts import load_prompt

print(f"Configured model: {MODEL}")
print("Package imports successful.")

Configured model: qwen3:8b
Package imports successful.


## Check the local Ollama environment

In [3]:
ollama_path = shutil.which("ollama")

if ollama_path is None:
    raise RuntimeError(
        "Ollama was not found in PATH. Install or start Ollama before continuing."
    )

version_result = subprocess.run(
    ["ollama", "--version"],
    capture_output=True,
    text=True,
    check=True,
)

models_result = subprocess.run(
    ["ollama", "list"],
    capture_output=True,
    text=True,
    check=True,
)

installed_models = {
    line.split()[0]
    for line in models_result.stdout.splitlines()[1:]
    if line.strip()
}

if MODEL not in installed_models:
    raise RuntimeError(
        f"Configured model '{MODEL}' is not installed. "
        f"Installed models: {sorted(installed_models)}"
    )

print(f"Ollama executable: {ollama_path}")
print(version_result.stdout.strip())
print(f"Model available: {MODEL}")

Ollama executable: C:\Users\hitch\AppData\Local\Programs\Ollama\ollama.EXE
ollama version is 0.32.1
Model available: qwen3:8b


## Test structured output

This small request verifies that `ask_llm()` can:

- reach Ollama
- request schema-constrained JSON
- validate the response with Pydantic


In [4]:
class Planet(BaseModel):
    name: str
    position_from_sun: int


class PlanetList(BaseModel):
    planets: list[Planet]


test_result = ask_llm(
    system_prompt=(
        "You are a precise educational assistant. "
        "Return valid JSON matching the requested schema."
    ),
    user_prompt="List the first three planets from the Sun in order.",
    schema=PlanetList,
    temperature=0.0,
    seed=42,
)

test_result

PlanetList(planets=[Planet(name='Mercury', position_from_sun=1), Planet(name='Venus', position_from_sun=2), Planet(name='Earth', position_from_sun=3)])

## Test prompt loading

In [5]:
test_prompt = load_prompt("test")

print(test_prompt)
print("\nPrompt loading successful.")

You are a helpful assistant.

Always answer in JSON.

Prompt loading successful.


## Foundation status

If all cells complete successfully, the local LLM foundation is ready for
the later pipeline notebooks.
